<a href="https://colab.research.google.com/github/Asad3ek/computer_vison/blob/main/menu_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
print('Menu Detector Model')

Menu Detector Model


In [19]:
#=======<<<<<>>>>=========
     #Import Libraries
#=======<<<<<>>>>=========
from google.colab import drive

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.models import mobilenet_v2

from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset, DataLoader
import os
import numpy as np

In [20]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
#Define dataset path

DATASET_PATH ='/content/drive/MyDrive/FOOD_101_dataset'
print('dataset_path: ', DATASET_PATH )


CUSTOM_CLASS_MAPPING = {
    "hamburger": "hamburger",
    "hot_dog": "hot_dog",
    "chocolate_cake": "dessert",  #Label grouping | class consolidation
    "cheesecake": "dessert",      #Label grouping | class consolidation
    "kebab": "kebab",
    "pilaf": "pilaf"
}

CLASSES = ['hamburger', 'hot_dog', 'dessert', 'kebab', 'pilaf']
CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)

print(CLASS_TO_IDX)

transform = transforms.Compose([
    transforms.Resize((224, 224)),                          #Before tensor: Height | Width | Channels [0 ~ 255]
    transforms.ToTensor(),                                  #channel: Red |Green | Blue
    transforms.Normalize(mean=[0.485, 0.456, 0.406],       #After tensor: Channels [0.0 ~ 1.0] | Height | Width
                         std=[0.229, 0.224, 0.225])        # Normalise is: pixel = (pixel - mean) / std

])

dataset_path:  /content/drive/MyDrive/FOOD_101_dataset
{'hamburger': 0, 'hot_dog': 1, 'dessert': 2, 'kebab': 3, 'pilaf': 4}


In [25]:
#========<<<<<<>>>>>==========
   #Custom Dataset Class
#========<<<<<<>>>>>==========


class FoodDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        print('images_length', len(self.images))
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        # print('image_path', img_path)
        label = self.labels[idx]
        # print('label', label)
        try:
            image = Image.open(img_path)
            if image.mode == "P" or image.mode == "RGBA":
              image = image.convert("RGBA").convert("RGB")
            else:
              image = image.convert("RGB")
        except (UnidentifiedImageError, OSError):
            print(f"Skipping broken image: {img_path}")
            return self.__getitem__((idx + 1) % len(self.images))
        if self.transform:
            image = self.transform(image)
        return image, label

In [26]:
#========<<<<<<>>>>>==========
   # Gather and Split Data
#========<<<<<<>>>>>==========

all_images = []
for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():
    class_path = os.path.join(DATASET_PATH, original_class)           # /content/drive/MyDrive/food101_dataset/hamburger
    # print('class_path:', class_path)
    if not os.path.exists(class_path):
        print(f"Warning: {class_path} not found")
        continue
    for img in os.listdir(class_path):
        if img.endswith(('.jpg', '.jpeg', '.png')): # .txt, .docx
            full_path = os.path.join(class_path, img)
            all_images.append((full_path, CLASS_TO_IDX[mapped_class]))

np.random.shuffle(all_images)
split = int(0.8 * len(all_images))
train_data = all_images[:split]
val_data = all_images[split:]

train_images, train_labels = zip(*train_data)
val_images, val_labels = zip(*val_data)

# print('all_images:', all_images)

dataset = FoodDataset(train_images, train_labels)
print(len(dataset))
img, lbl = dataset[0]

images_length 2400
2400


In [27]:
train_dataset = FoodDataset(train_images, train_labels, transform=transform)
val_dataset = FoodDataset(val_images, val_labels, transform=transform)

In [28]:
train_loader = DataLoader(train_dataset, batch_size=30, shuffle=True, num_workers=2)  #thred | parallel loading for speed
val_loader = DataLoader(val_dataset, batch_size=30, shuffle=False, num_workers=2)

images_length 2400
images_length 2400


In [29]:
model = mobilenet_v2(weights='IMAGENET1K_V1')   #pretrained model | lightweit | CNN | 1000 classes
model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)    #fine-tuning | backbone | model layer freeze

In [30]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE: ', device)
model = model.to(device)

DEVICE:  cuda


In [31]:
criterion = nn.CrossEntropyLoss()                           #Loss function | burger: 70%.. "calculatehow much predection true or no"
optimizer =optim.Adam(model.parameters(), lr=0.001)         #weight
torch.backends.cudnn.benchmark = True

In [32]:
#========<<<<<<>>>>>==========
   # Training Loop
#========<<<<<<>>>>>==========

NUM_EPOCHS = 10
best_accuracy = 0.0

for epoch in range(NUM_EPOCHS):
    model.train()  # train mode
    running_loss = 0.0  # 70% | 30%Loss | 100%
    for images, labels in train_loader:  # Forward and Backward(Backpropagation)
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()  # zero the gradient
        outputs = model(images)  # Forward Pass | Dog | 5 Classes
        loss = criterion(outputs, labels)  # Calculate Loss
        loss.backward()
        optimizer.step()  # Adam optimizer
        running_loss += loss.item()  # Track Loss

# Validation
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

val_acc = 100 * correct / total
print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] Loss: {running_loss/len(train_loader):.4f}, Val Accuracy: {val_acc:.2f}%")

if val_acc > best_accuracy:
    best_accuracy = val_acc
    torch.save(model.state_dict(), '/content/menu_detector.pth')
    print("Saved new best model!")

images_length 2400
images_length 2400
images_length 2400
images_length 2400
images_length 2400
images_length 2400
images_length 2400
images_length 2400
images_length 2400
images_length 2400
images_length 2400
images_length 2400
images_length 2400
images_length 2400
images_length 2400
images_length 2400
images_length 2400
images_length 2400
images_length 2400
images_length 2400
images_length 2400
images_length 2400
images_length 2400
images_length 2400
images_length 2400
images_length 2400
images_length 2400
images_length 2400
images_length 2400
images_length 2400
images_length 600
images_length 2400
Epoch [10/10] Loss: 0.0907, Val Accuracy: 88.50%
Saved new best model!
